In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct", # Changed model name to a valid Unsloth-supported HuggingFace model
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def clean(example):
  messages = example["messages"]

  messages = [m for m in messages if m["content"].strip() != ""]
  return {"messages": messages}

dataset = dataset.map(clean)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/llama-3-8b-Instruct")

def format_chat(example):
  return{
      "text":tokenizer.apply_chat_template(
          example["messages"],
          tokenize=False
      )
  }

dataset = dataset.map(format_chat)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
max_length=512

def tokenize(example):
  return tokenizer(
      example["text"],
      truncation=True,
      padding='max_length',
      max_length=max_length
  )

dataset = dataset.map(tokenize)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from unsloth import FastLanguageModel

# Prepare model for PEFT training by adding LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0, # Currently only supports dropout = 0
    bias = "none",    # Currently only supports bias = "none"
    use_gradient_checkpointing = "unsloth", # Changed from "current" to "unsloth"
    random_state = 3407,
    max_seq_length = max_length, # Uses max_length from a previous cell
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    # max_seq_length=512, # Removed this line as it's not supported by Unsloth's SFTTrainer
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Unsloth 2026.3.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 3 | Total steps = 3,750
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,11.713974
2,11.712421
3,11.719451
4,11.707891
5,11.706676
6,11.698878
7,11.711019
8,11.681633
9,11.705290
10,11.722273


Step,Training Loss
1,11.713974
2,11.712421
3,11.719451
4,11.707891
5,11.706676
6,11.698878
7,11.711019
8,11.681633
9,11.705290
10,11.722273


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("my_model")
tokenizer.save_pretrained("my_model")

('my_model/tokenizer_config.json',
 'my_model/chat_template.jinja',
 'my_model/tokenizer.json')

In [ ]:
import shutil
shutil.make_archive("my_model", 'zip', "my_model")

'/content/my_model.zip'

In [ ]:
from google.colab import files
files.download("my_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.save_pretrained_gguf(
    "model",
    tokenizer,
    quantization_method="q4_0"
)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 26462.49it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:32<00:00, 128.08s/it]


Unsloth: Merge process complete. Saved to `/content/model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf/llama-3-8b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf/llama-3-8b-instruct.Q4_0.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model model_gguf/llama-3-8b-instruct.Q4_0.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f model_gguf/Modelfile


{'save_directory': 'model',
 'gguf_directory': 'model_gguf',
 'gguf_files': ['model_gguf/llama-3-8b-instruct.Q4_0.gguf'],
 'modelfile_location': 'model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
from google.colab import files
files.download("model_gguf/llama-3-8b-instruct.Q4_0.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>